In [1]:
import sys
sys.path.insert(0, '.')

from embedder import Embedder

embedder = Embedder()

query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

print(f"Vector length: {len(v)}")
print(f"First value v[0]: {v[0]}")

Vector length: 384
First value v[0]: -0.02058203437252893


In [2]:
import numpy as np
from gitsource import GithubRepositoryDataReader

# load documents
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print(f"Total documents: {len(documents)}")

# find the specific document
target = next(d for d in documents if d['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md')
print(f"Found: {target['filename']}")

# embed its content
doc_vector = embedder.encode(target['content'])

# cosine similarity — vectors are normalized so dot product = cosine similarity
similarity = np.dot(v, doc_vector)
print(f"Cosine similarity: {similarity:.4f}")

Total documents: 72
Found: 02-vector-search/lessons/07-sqlitesearch-vector.md
Cosine similarity: 0.3611


In [3]:
from gitsource import chunk_documents
from tqdm import tqdm

# chunk documents
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total chunks: {len(chunks)}")

# embed all chunks
print("Embedding chunks...")
contents = [chunk['content'] for chunk in chunks]
X = embedder.encode_batch(contents)
X = np.array(X)
print(f"Matrix shape: {X.shape}")

# score query against all chunks
scores = X.dot(v)

# find highest scoring chunk
best_idx = np.argmax(scores)
best_chunk = chunks[best_idx]

print(f"\nBest score: {scores[best_idx]:.4f}")
print(f"Best chunk filename: {best_chunk['filename']}")

Total chunks: 295
Embedding chunks...
Matrix shape: (295, 384)

Best score: 0.6489
Best chunk filename: 02-vector-search/lessons/07-sqlitesearch-vector.md


In [6]:


from minsearch import VectorSearch

vector_index = VectorSearch(
    keyword_fields=["filename"]
)
vector_index.fit(X, chunks)  # vectors first, payload second

# search
query4 = "What metric do we use to evaluate a search engine?"
v4 = embedder.encode(query4)

results = vector_index.search(v4, num_results=5)

print("Top 5 results:")
for r in results:
    print(f"- {r['filename']}")

Top 5 results:
- 04-evaluation/lessons/05-search-metrics.md
- 04-evaluation/lessons/01-intro.md
- 01-agentic-rag/lessons/05-search.md
- 04-evaluation/lessons/01-intro.md
- 04-evaluation/lessons/15-next-steps.md


In [5]:
help(VectorSearch.fit)

Help on function fit in module minsearch.vector:

fit(self, vectors, payload)
    Fits the index with the provided vectors and payload documents.

    Args:
        vectors (np.ndarray): 2D numpy array of shape (n_docs, vector_dimension).
        payload (list of dict): List of documents to use as payload. Each document is a dictionary.



In [7]:
from minsearch import Index

# keyword index
text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
text_index.fit(chunks)

query5 = "How do I store vectors in PostgreSQL?"
v5 = embedder.encode(query5)

# vector search top 5
vector_results = vector_index.search(v5, num_results=5)
vector_filenames = [r['filename'] for r in vector_results]

# text search top 5
text_results = text_index.search(query5, num_results=5)
text_filenames = [r['filename'] for r in text_results]

print("Vector results:")
for f in vector_filenames:
    print(f"  - {f}")

print("\nText results:")
for f in text_filenames:
    print(f"  - {f}")

print("\nIn vector but NOT in text:")
for f in vector_filenames:
    if f not in text_filenames:
        print(f"  - {f}")

Vector results:
  - 02-vector-search/lessons/08-pgvector.md
  - 02-vector-search/lessons/08-pgvector.md
  - 03-orchestration/lessons/05-rag.md
  - 02-vector-search/lessons/08-pgvector.md
  - 02-vector-search/lessons/08-pgvector.md

Text results:
  - 02-vector-search/lessons/02-embeddings.md
  - 03-orchestration/lessons/05-rag.md
  - 02-vector-search/lessons/01-intro.md
  - 03-orchestration/lessons/05-rag.md
  - 02-vector-search/lessons/01-intro.md

In vector but NOT in text:
  - 02-vector-search/lessons/08-pgvector.md
  - 02-vector-search/lessons/08-pgvector.md
  - 02-vector-search/lessons/08-pgvector.md
  - 02-vector-search/lessons/08-pgvector.md


In [8]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

query6 = "How do I give the model access to tools?"
v6 = embedder.encode(query6)

vector_results = vector_index.search(v6, num_results=5)
text_results = text_index.search(query6, num_results=5)

fused = rrf([vector_results, text_results])

print("RRF results:")
for r in fused:
    print(f"- {r['filename']}")

RRF results:
- 01-agentic-rag/lessons/13-function-calling.md
- 01-agentic-rag/lessons/01-intro.md
- 01-agentic-rag/lessons/14-agentic-loop.md
- 04-evaluation/lessons/02-ground-truth.md
- 01-agentic-rag/lessons/16-other-frameworks.md
